In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 2 e o Qiskit SDK

<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar estas versões ou mais recentes.

```
qiskit[all]~=2.3.0
```
</details>

O Qiskit SDK fornece algumas ferramentas para converter entre representações OpenQASM de programas quânticos e a classe [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit).

<span id="qasm2-import"></span>
## Importar um programa OpenQASM 2 para o Qiskit
Duas funções importam programas OpenQASM 2 para o Qiskit.
São elas: [`qasm2.load()`](../api/qiskit/qasm2#load), que recebe um nome de arquivo, e [`qasm2.loads()`](../api/qiskit/qasm2#loads), que recebe o programa OpenQASM 2 como uma string.

In [1]:
import qiskit.qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";
    qreg q[2];
    creg c[2];

    h q[0];
    cx q[0], q[1];

    measure q -> c;
"""
circuit = qiskit.qasm2.loads(program)
circuit.draw()

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

Consulte a [API OpenQASM 2 do Qiskit](https://docs.quantum.ibm.com/api/qiskit/qasm2) para mais informações.

### Importar programas simples
Para a maioria dos programas OpenQASM 2, você pode usar simplesmente `qasm2.load` e `qasm2.loads` com um único argumento.

#### Exemplo: importar um programa OpenQASM 2 como string
Use `qasm2.loads()` para importar um programa OpenQASM 2 como string em um QuantumCircuit:

In [2]:
from qiskit import qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";

    qreg q[4];
    creg c[4];

    h q[0];
    cx q[0], q[1];

    // 'rxx' is not actually in `qelib1.inc`,
    // but Qiskit used to behave as if it were.
    rxx(0.75) q[2], q[3];

    measure q -> c;
"""
circuit = qasm2.loads(
    program,
    custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
)

#### Example: use a particular gate class when importing an OpenQASM 2 program

Qiskit cannot, in general, verify if the definition in an OpenQASM 2 `gate` statement corresponds exactly to a Qiskit standard-library gate.
Instead, Qiskit chooses a custom gate using the precise definition supplied.
This can be less efficient that using one of the built-in standard gates, or a user-defined custom gate.
You can manually define `gate` statements with particular classes.

In [3]:
from qiskit import qasm2
from qiskit.circuit import Gate
from qiskit.circuit.library import RZXGate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    # Link the OpenQASM 2 name 'my' with our custom gate.
    qasm2.CustomInstruction("my", 2, 1, MyGate),
    # Link the OpenQASM 2 name 'rzx' with Qiskit's
    # built-in RZXGate.
    qasm2.CustomInstruction("rzx", 1, 2, RZXGate),
]

program = """
    OPENQASM 2.0;

    gate my(theta, phi) q {
        U(theta / 2, phi, -theta / 2) q;
    }
    gate rzx(theta) a, b {
        // It doesn't matter what definition is
        // supplied, if the parameters match;
        // Qiskit will still use `RZXGate`.
    }

    qreg q[2];
    my(0.25, 0.125) q[0];
    rzx(pi) q[0], q[1];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

#### Exemplo: importar um programa OpenQASM 2 a partir de um arquivo
Use `load()` para importar um programa OpenQASM 2 de um arquivo em um QuantumCircuit:

In [4]:
from qiskit import qasm2
from qiskit.circuit import Gate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    qasm2.CustomInstruction("my", 2, 1, MyGate, builtin=True),
]

program = """
    OPENQASM 2.0;
    qreg q[1];

    my(0.25, 0.125) q[0];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

<span id="custom-instructions"></span>
### Vincular gates OpenQASM 2 com gates do Qiskit
Por padrão, o importador OpenQASM 2 do Qiskit trata o arquivo de inclusão `"qelib1.inc"` como uma biblioteca padrão *de facto*.
O importador trata este arquivo como contendo exatamente os gates descritos no [artigo original que define o OpenQASM 2](https://arxiv.org/abs/1707.03429).
O Qiskit usará os gates integrados da [biblioteca de circuitos](../api/qiskit/circuit_library) para representar os gates em `"qelib1.inc"`.
Gates definidos no programa por instruções `gate` manuais do OpenQASM 2 serão, por padrão, construídos como [subclasses de `Gate` do Qiskit](../api/qiskit/qiskit.circuit.Gate) personalizadas.

Você pode instruir o importador a usar classes [`Gate`](../api/qiskit/qiskit.circuit.Gate) específicas para as instruções `gate` encontradas.
Você também pode usar este mecanismo para tratar nomes de gates adicionais como "integrados", ou seja, sem exigir uma definição explícita.
Se você especificar quais classes de gate usar para instruções `gate` fora de `"qelib1.inc"`, o circuito resultante normalmente será mais eficiente para trabalhar.

> **Warning:** A partir do Qiskit SDK v1.0, o *exportador* OpenQASM 2 do Qiskit (veja [Exportar um circuito Qiskit para OpenQASM 2](#qasm2-export)) ainda se comporta como se `"qelib1.inc"` tivesse mais gates do que realmente tem.
> Isso significa que as configurações padrão do importador podem não conseguir importar um programa exportado pelo nosso exportador.
> Veja [o exemplo específico sobre como trabalhar com o exportador legado](#qasm2-import-legacy) para resolver este problema.
> 
> Esta discrepância é um comportamento legado do Qiskit, e [será resolvida em uma versão futura do Qiskit](https://github.com/Qiskit/qiskit/issues/10737).

Para passar informações sobre uma instrução personalizada ao importador OpenQASM 2, use [a classe `qasm2.CustomInstruction`](../api/qiskit/qasm2#qiskit.qasm2.CustomInstruction).
Esta classe requer quatro informações, em ordem:

* O **nome** do gate, usado no programa OpenQASM 2
* O **número de parâmetros de ângulo** que o gate recebe
* O **número de qubits** sobre os quais o gate atua
* A classe ou função Python **construtora** do gate, que recebe os parâmetros do gate (mas não os qubits) como argumentos individuais

Se o importador encontrar uma definição `gate` que corresponda a uma instrução personalizada fornecida, usará essas informações personalizadas para reconstruir o objeto gate.
Se uma instrução `gate` for encontrada com o `name` de uma instrução personalizada, mas sem corresponder ao número de parâmetros e ao número de qubits, o importador lançará um [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror), indicando a incompatibilidade entre as informações fornecidas e o programa.

Além disso, um quinto argumento `builtin` pode ser definido opcionalmente como `True` para tornar o gate automaticamente disponível no programa OpenQASM 2, mesmo que não seja definido explicitamente.
Se o importador encontrar uma definição explícita de `gate` para uma instrução personalizada integrada, ela será aceita silenciosamente.
Da mesma forma, se uma definição explícita do mesmo nome não for compatível com a instrução personalizada fornecida, um [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror) será lançado.
Isso é útil para compatibilidade com exportadores OpenQASM 2 mais antigos e com certas outras plataformas quânticas que tratam os "gates de base" de seu hardware como instruções integradas.

O Qiskit fornece um atributo de dados para trabalhar com programas OpenQASM 2 produzidos por versões legadas das [capacidades de exportação OpenQASM 2 do Qiskit](#qasm2-export).
Este é o [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility), que pode ser passado como argumento `custom_instructions` para [`qasm2.load()`](../api/qiskit/qasm2#load) e [`qasm2.loads()`](../api/qiskit/qasm2#loads).

<span id="qasm2-import-legacy"></span>
#### Exemplo: importar um programa criado pelo exportador legado do Qiskit
Este programa OpenQASM 2 usa gates que não estão na versão original de `"qelib1.inc"` sem declará-los, mas que são gates padrão na biblioteca do Qiskit.
Você pode usar [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility) para facilmente instruir o importador a usar o mesmo conjunto de gates que o exportador OpenQASM 2 do Qiskit usava anteriormente.

In [5]:
import math
from qiskit import qasm2

program = """
    include "qelib1.inc";
    qreg q[2];
    rx(arctan(pi, 3 + add_one(0.2))) q[0];
    cx q[0], q[1];
"""


def add_one(x):
    return x + 1


customs = [
    # Our `add_one` takes only one parameter.
    qasm2.CustomClassical("add_one", 1, add_one),
    # `arctan` takes two parameters, and `math.atan2` implements it.
    qasm2.CustomClassical("arctan", 2, math.atan2),
]
circuit = qasm2.loads(program, custom_classical=customs)

#### Exemplo: usar uma classe de gate específica ao importar um programa OpenQASM 2
O Qiskit não consegue, em geral, verificar se a definição em uma instrução `gate` do OpenQASM 2 corresponde exatamente a um gate da biblioteca padrão do Qiskit.
Em vez disso, o Qiskit escolhe um gate personalizado usando a definição precisa fornecida.
Isso pode ser menos eficiente do que usar um dos gates padrão integrados ou um gate personalizado definido pelo usuário.
Você pode definir manualmente instruções `gate` com classes específicas.

In [6]:
from qiskit import QuantumCircuit, qasm2

# Define any circuit.
circuit = QuantumCircuit(2, 2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure([0, 1], [0, 1])

# Export to a string.
program = qasm2.dumps(circuit)

# Export to a file.
qasm2.dump(circuit, "my_file.qasm")

#### Exemplo: definir um novo gate integrado em um programa OpenQASM 2
Se o argumento `builtin=True` for definido, um gate personalizado não precisa ter uma definição associada.